# Session 02: Practice Quiz

**Notebook 2 of 3: Self-check** · about 35 minutes

Do this after both `01-concepts.ipynb` and `03-collections-deep-dive.ipynb`. For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

## Q1 · Predict the output

```python
on_call = {"team": "platform", "engineers": ["amira"]}
weekend = on_call
weekend["engineers"].append("jonas")
weekend = {"team": "platform", "engineers": ["li"]}
print(on_call["engineers"])
```

Write your prediction, then run it.

In [ ]:
on_call = {"team": "platform", "engineers": ["amira"]}
weekend = on_call
weekend["engineers"].append("jonas")
weekend = {"team": "platform", "engineers": ["li"]}
print(on_call["engineers"])

<details>
<summary>Show answer</summary>

`['amira', 'jonas']`.

- Line 2 attaches a second name to the same dict, so the `append` on line 3 changes the list that `on_call` sees.
- Line 4 **rebinds** `weekend` to a brand-new dict. Rebinding a name never affects the object it used to point at, so `on_call` is untouched by it.

Two different operations: *mutating an object* (visible through every name) versus *rebinding a name* (affects only that name).

</details>

## Q2 · Predict the output

```python
defaults = ["network"]
ticket_a = {"id": "INC-1", "tags": defaults}
ticket_b = {"id": "INC-2", "tags": defaults.copy()}
ticket_a["tags"] += ["dns"]
print(defaults, ticket_b["tags"])
```

In [ ]:
defaults = ["network"]
ticket_a = {"id": "INC-1", "tags": defaults}
ticket_b = {"id": "INC-2", "tags": defaults.copy()}
ticket_a["tags"] += ["dns"]
print(defaults, ticket_b["tags"])

<details>
<summary>Show answer</summary>

`['network', 'dns'] ['network']`.

`ticket_a["tags"]` *is* the `defaults` list, and `+=` on a list extends it **in place**, so `defaults` changes too. `ticket_b` got its own copy, so it's unaffected.

If you predicted `defaults` would stay `['network']`, you were probably thinking of `x = x + [...]`, which builds a new list. For lists, `+=` and `= ... +` are not the same operation.

</details>

## Q3 · Predict the output

```python
records = ["INC-1", "INC-2", "INC-3", "INC-4", "INC-5"]
print(records[1:3], records[-2:], records[::-2], records[10:])
```

<details>
<summary>Show answer</summary>

`['INC-2', 'INC-3'] ['INC-4', 'INC-5'] ['INC-5', 'INC-3', 'INC-1'] []`

- `[1:3]`: start included, stop excluded, so positions 1 and 2.
- `[-2:]`: the last two.
- `[::-2]`: walk backwards from the end, taking every second item.
- `[10:]`: out-of-range slices return an empty list rather than raising. `records[10]` (indexing, not slicing) would raise `IndexError`.

</details>

## Q4 · Debugging: the missing escalations

This function should return the IDs of tickets with **no escalation level set**. The data uses `0` for "escalated to level 0, the duty engineer".

```python
def needs_escalation_level(tickets):
    return [t["id"] for t in tickets if not t["escalation_level"]]

tickets = [
    {"id": "INC-1", "escalation_level": 0},
    {"id": "INC-2", "escalation_level": None},
    {"id": "INC-3", "escalation_level": 2},
]
print(needs_escalation_level(tickets))   # expected: ['INC-2']
```

What does it actually print, why, and what is the fix?

<details>
<summary>Show answer</summary>

It prints `['INC-1', 'INC-2']`.

`not 0` is `True`, because `0` is falsy, so a valid level of zero is treated as missing. Truthiness merges "missing" with "zero or empty".

Fix: test for the missing value explicitly:

```python
return [t["id"] for t in tickets if t["escalation_level"] is None]
```

**Rule:** use truthiness only when empty and missing genuinely mean the same thing.

</details>

## Q5 · Comparison: TypeScript habits

A TypeScript developer writes:

```python
def notify(recipients):
    if recipients:
        print("notifying", recipients)
    else:
        print("no recipients")

notify([])
```

In TypeScript, `if (recipients)` with an empty array takes the first branch. What does Python print, and why does the difference matter?

<details>
<summary>Show answer</summary>

Python prints `no recipients`.

In JavaScript and TypeScript, `[]` and `{}` are **truthy**: they're objects, and all objects are truthy. In Python, **empty collections are falsy**. So `if items:` in Python means "if there is at least one item", which is usually what you meant anyway. But a habit carried over from TypeScript, such as `if recipients is not None and len(recipients) > 0`, is just noise in Python, and the opposite assumption can create real bugs.

</details>

## Q6 · Predict the output: `is` versus `==`

```python
expected = ["P1", "P2"]
received = ["P1", "P2"]
print(expected == received, expected is received)

status = None
print(status is None, status == None)
```

<details>
<summary>Show answer</summary>

```text
True False
True True
```

- The lists have equal **values** but are two separate **objects**.
- For `None`, both forms work, but `is None` is the idiom. `None` is a singleton (there is exactly one `None` object), so identity is the precise check. `==` can also be overridden by a class to return something surprising. Ruff flags `== None` (rule E711).

Use `==` for values; use `is` for `None`, `True`/`False` singletons in rare cases, and your own sentinel objects.

</details>

## Q7 · Debugging: a cleanup loop

```python
tags = ["temp", "network", "temp", "temp", "dns"]
for tag in tags:
    if tag == "temp":
        tags.remove(tag)
print(tags)
```

The author expected `['network', 'dns']`. Run it. What happened, and how would you fix it?

In [ ]:
tags = ["temp", "network", "temp", "temp", "dns"]
for tag in tags:
    if tag == "temp":
        tags.remove(tag)
print(tags)

<details>
<summary>Show answer</summary>

It prints `['network', 'temp', 'dns']`: one `"temp"` survives.

The loop advances by position. Each `remove()` shifts the remaining items left, so the item that slides into the current position is skipped. Python raises no error for this with lists; it just gives a wrong answer.

Fix: build a new list instead of editing the one you're looping over:

```python
tags = [tag for tag in tags if tag != "temp"]
```

</details>

## Q8 · Implementation: comprehensions

Using the `tickets` list below, write **one comprehension each** for:

1. `breached_ids`: IDs of tickets whose `age_hours` exceeds the SLA for their priority (use `SLA_HOURS`).
2. `teams`: the set of distinct teams.
3. `age_by_id`: a dict mapping ticket ID to age in hours.

In [ ]:
SLA_HOURS = {"P1": 4, "P2": 24, "P3": 72}
tickets = [
    {"id": "INC-1", "team": "platform", "priority": "P1", "age_hours": 6},
    {"id": "INC-2", "team": "database", "priority": "P2", "age_hours": 10},
    {"id": "INC-3", "team": "platform", "priority": "P3", "age_hours": 90},
]

breached_ids = None   # replace
teams = None          # replace
age_by_id = None      # replace

print(breached_ids, teams, age_by_id)

<details>
<summary>Show answer</summary>

```python
breached_ids = [t["id"] for t in tickets if t["age_hours"] > SLA_HOURS[t["priority"]]]
teams = {t["team"] for t in tickets}
age_by_id = {t["id"]: t["age_hours"] for t in tickets}
```

Expected: `['INC-1', 'INC-3']`, `{'platform', 'database'}` (in some order, since sets are unordered), and `{'INC-1': 6, 'INC-2': 10, 'INC-3': 90}`.

Note how the brackets choose the result type: `[...]` list, `{x ...}` set, `{k: v ...}` dict.

</details>

## Q9 · Debugging: shared lists

This code should group ticket IDs by team. Every team ends up with every ID. Why?

```python
teams = ["platform", "database"]
groups = dict.fromkeys(teams, [])
groups["platform"].append("INC-1")
groups["database"].append("INC-2")
print(groups)
```

<details>
<summary>Show answer</summary>

Output: `{'platform': ['INC-1', 'INC-2'], 'database': ['INC-1', 'INC-2']}`.

`dict.fromkeys(teams, [])` creates **one** empty list and stores that same object under every key. You can confirm it with `groups["platform"] is groups["database"]`, which prints `True`.

Fix: create a new list per key:

```python
groups = {team: [] for team in teams}
# or
from collections import defaultdict
groups = defaultdict(list)
```

The same trap appears as `[[]] * 3`, which is three references to one inner list.

</details>

## Q10 · Engineering judgment: choose the collection

For each situation, which collection would you use, and why?

1. Ticket IDs already processed in this run, checked once for each of 200,000 incoming events.
2. The on-call rota for the week, Monday to Sunday, in order.
3. A lookup from team name to its Slack channel.
4. A `(team, priority)` pair used to count breaches.
5. The tags on one ticket, where duplicates are meaningless and you often ask "does it have tag X?".

<details>
<summary>Show answer</summary>

1. **set**: the only operation is "seen it?", 200,000 times. A list would scan on every check.
2. **list** (or a tuple if it never changes during the run): order is the point, and positions mean days.
3. **dict**: lookup by key.
4. **tuple**: a fixed-shape pair that has to be **hashable** to serve as a dict key. A list can't be a key.
5. **set**: uniqueness and membership. If you need a stable order when displaying them, `sorted(tags)` at output time.

The selection rule from the deep-dive notebook: start from **how the data is accessed**, not what it contains.

</details>

## Q11 · Predict the output: sorting

```python
priorities = ["P3", "P1", "P2"]
ranked = priorities.sort()
print(ranked, priorities)

tickets = [("INC-1", "P2", 5), ("INC-2", "P1", 9), ("INC-3", "P2", 30)]
print([t[0] for t in sorted(tickets, key=lambda t: (t[1], -t[2]))])
```

<details>
<summary>Show answer</summary>

```text
None ['P1', 'P2', 'P3']
['INC-2', 'INC-3', 'INC-1']
```

- `list.sort()` sorts **in place** and returns `None`, so `ranked` is `None`. Use `sorted()` when you want a new list.
- The key `(priority, -age)` sorts by priority first (`P1` before `P2`), then by age **descending** within a priority, because negating a number reverses its order. `INC-3` (30 hours) comes before `INC-1` (5 hours).

</details>

## Q12 · Implementation: group and aggregate

Write `breaches_by_team(tickets, sla_hours)`. It returns a dict mapping each team to the **number** of its tickets that breached their SLA. Teams with no breaches must still appear, with `0`. It must not modify `tickets`.

```python
expected = {"platform": 2, "database": 0}
```

In [ ]:
SLA_HOURS = {"P1": 4, "P2": 24, "P3": 72}
tickets = [
    {"id": "INC-1", "team": "platform", "priority": "P1", "age_hours": 6},
    {"id": "INC-2", "team": "database", "priority": "P2", "age_hours": 10},
    {"id": "INC-3", "team": "platform", "priority": "P3", "age_hours": 90},
]


def breaches_by_team(tickets, sla_hours):
    ...  # your implementation


print(breaches_by_team(tickets, SLA_HOURS))

<details>
<summary>Show answer</summary>

One readable solution:

```python
def breaches_by_team(tickets, sla_hours):
    counts = {ticket["team"]: 0 for ticket in tickets}   # every team starts at 0
    for ticket in tickets:
        if ticket["age_hours"] > sla_hours[ticket["priority"]]:
            counts[ticket["team"]] += 1
    return counts
```

Why this shape:

- Seeding every team with `0` first is what makes "teams with no breaches still appear" work. A `Counter` or `defaultdict(int)` would only contain teams that had at least one breach.
- It reads `tickets` but never changes it, and it builds and returns a new dict.
- `sla_hours[ticket["priority"]]` uses `[]`, not `.get()`: an unknown priority is a data error. `[]` fails immediately with `KeyError: 'P9'`, which names the bad value. `.get()` would return `None`, and the comparison would then fail with `TypeError: '>' not supported between instances of 'int' and 'NoneType'`, which tells you much less.

A compact alternative is `{team: sum(...) for team in {t["team"] for t in tickets}}`, but it loops over the tickets once per team. Fine for small data, wasteful for large.

</details>

## How did you do?

- **Q1, Q2, Q9** cover names, aliasing and mutability. If any surprised you, re-read Parts 1 and 2 of `01-concepts.ipynb`.
- **Q4–Q6** cover truthiness, `None` and `is`: Part 4.
- **Q3, Q7, Q8, Q11** cover slicing, loops, comprehensions and sorting.
- **Q10, Q12** cover choosing and using collections: `03-collections-deep-dive.ipynb`.

Next: the workbook assignment in [`../workbook/assignment.md`](../workbook/assignment.md). Q12 is a small preview of it.